# Lesson 20 | Before loading a real MaleCNS subset: verify the network image

The project eventually needs software and FPGA to consume the **same, traceable** converted network image.

Today asks one question:

> **Before a real MaleCNS subset is loaded, how can we prove that two consumers received the same versioned bytes with explicit provenance?**

Primary new concept: **manifest + checksum + provenance as an image-integrity contract.**

Important boundary: the formal RMD-017/018 MaleCNS artifact is not yet declared complete. This lesson uses a tiny teaching fixture to learn the image contract; it does **not** claim that a real MaleCNS subset has been loaded.

## 1. Concept ledger

**Already known:** connectome nodes/edges/metadata, binary data movement, host control, and the **Field-Programmable Gate Array (FPGA)** side of the system.

**New today:** **manifest** (a small description of an artifact) and **checksum** (a deterministic digest of exact bytes). This lesson uses **Secure Hash Algorithm 256-bit (SHA-256)** as the concrete checksum algorithm.

**Preview only:** real MaleCNS conversion, formal binary schema, 1K differential test, and full dataset loading.

## 2. What must travel together?

A reproducible network artifact needs more than a file name. This teaching manifest contains at least:

- `schema_version`: which schema interprets the bytes;
- `source_release`: which source release / fixture the data came from;
- `converter_version`: which converter version produced the image;
- `byte_count`: exact byte length;
- `sha256`: checksum of the exact image bytes.

Here, provenance is made explicit by `source_release + converter_version`.

A checksum answers **“are these bytes identical?”**  
Provenance answers **“where did these bytes come from, and under which conversion rule?”**

Neither answer **“is this scientifically the correct network?”**

## 3. Conversion and replay picture

<div style="max-width:860px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 860 420" role="img" aria-label="versioned network image integrity flow" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="l20-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2f5f3f"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="21" text-anchor="middle">
    <rect x="35" y="155" width="190" height="78" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="130" y="187" fill="#1f2d24">source data</text><text x="130" y="214" fill="#1f2d24">+ release ID</text>
    <rect x="285" y="155" width="190" height="78" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="380" y="187" fill="#1f2d24">versioned</text><text x="380" y="214" fill="#1f2d24">converter</text>
    <rect x="535" y="60" width="240" height="78" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="655" y="108" fill="#1f2d24">binary image</text>
    <rect x="535" y="250" width="240" height="92" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="655" y="282" fill="#1f2d24">manifest</text><text x="655" y="309" fill="#1f2d24">version + provenance</text><text x="655" y="336" fill="#1f2d24">byte count + SHA-256</text>
    <rect x="650" y="155" width="160" height="58" rx="8" fill="#f4fbf6" stroke="#3f7a50" stroke-width="2"/>
    <text x="730" y="191" fill="#1f2d24">consumers</text>
  </g>
  <path d="M225 194 L285 194" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
  <path d="M475 178 C500 150,520 125,548 112" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
  <path d="M475 210 C505 235,520 260,548 278" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
  <path d="M655 138 L700 155" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
  <path d="M655 250 L700 213" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
</svg>
</div>

## 4. Run: make and verify a teaching image

The payload below is just a byte fixture chosen so the integrity workflow is runnable without network access or a formal MaleCNS release artifact.

In [ ]:
import hashlib
import json

payload = bytes([19, 20, 1, 0, 3, 2, 7, 11])
manifest = {
    "schema_version": "teaching-schema-v1",
    "source_release": "teaching-fixture-v1",
    "converter_version": "teaching-converter-v1",
    "byte_count": len(payload),
    "sha256": hashlib.sha256(payload).hexdigest(),
}

software_bytes = bytes(payload)
fpga_replay_bytes = bytes(payload)

print(json.dumps(manifest, indent=2))
print("software checksum:", hashlib.sha256(software_bytes).hexdigest())
print("FPGA replay checksum:", hashlib.sha256(fpga_replay_bytes).hexdigest())
print("same image:", software_bytes == fpga_replay_bytes)

## 5. Observe

Both consumers receive identical bytes and therefore the same SHA-256 checksum; the manifest also records the source release and converter version explicitly.

This is still an **integrity + provenance proof**, not yet a neural correctness proof. T-015 later compares the behavior produced by the real subset in software and FPGA.

## 6. Why version the converter?

If the conversion rule changes — for example edge ordering, weight encoding, or record width — the same scientific source dataset may produce different binary bytes.

That is why the converter/schema version belongs in the manifest. Reproducibility needs the **data version and conversion rule**, not only the source name.

## 7. Differential test comes after integrity

A useful order is:

1. verify manifest and checksum;
2. load the same image into both implementations;
3. replay the same initial state and input events;
4. compare outputs/state using the approved oracle.

If step 1 fails, behavioral differences are impossible to interpret cleanly.

## 8. Try It

First change one payload byte and recompute the digest. Which manifest fields change?

Then keep the payload unchanged and change only `converter_version` from `v1` to `v2`. Why can the checksum stay the same while the provenance contract changes?

## 9. Exercise

[Lesson 20 exercise: build an image manifest with provenance](../../exercises/en/20_load_malecns_subset.ipynb)

## 10. AI Task

Ask an AI for a proposed manifest schema. Mark each field as integrity, provenance, schema/version, or experiment state. Reject fields that silently mix runtime neuron state into the static connectome image.

## 11. Human Check

Explain what checksum, provenance, and neural correctness each establish. Why must software and FPGA differential tests consume the exact same binary image and record the same source/converter versions?

## 12. Engineering Handoff

Maps to `RMD-017 / RMD-018`, `MOD-011`, and `T-014 / T-015`. The formal engineering slice must replace the teaching fixture with a documented MaleCNS-derived artifact before claiming a real-subset result.

## 13. Project Trace

- Lesson: `LSN-020`
- Mapping: `RMD-018` with prerequisite `RMD-017`
- Requirement path: `TRACE-C-001`
- Integrity oracle: `T-014`
- Real-subset behavioral oracle: `T-015`

## 14. Exit Ticket

Given a binary image and manifest, you can verify byte count and checksum, explain what that proves, and state what still requires a differential behavioral test.